In [18]:
from tqdm.notebook import tqdm
import polars as pl
from itertools import product

# wags-llm imports
from wags_llm.cache import InMemoryCache
from wags_llm.client import BedrockClaudeJsonClient
from wags_llm.prompts import build_empty_registry
from wags_llm.services import StructuredTaskRunner

# dgiLIT Pydantic Models
from dgilit_models import InteractionClassificationPrompt
from dgilit_models import InteractionClassificationResult

from dgilit_wags.tagging import (
    BioBertEntityTagger,
    CompositeEntityTagger,
    EntityPreTaggingService,
    PubTator3ChemicalTagger,
    SQLiteNormalizerCache,
    TaggerConfig,
    ViccNormalizer,
)


# wags-llm
Wags-LLM exists to add structure to LLM prompt calls in a repeatable, explainable way. This is the first attempt at implementing this with a dgiLIT prompt. Other portions of the dgiLIT pipeline would happen before this step and as such, the supporting models and prompts in this branch will likely evolve as those are tuned up.  
  
Notable things to update:  
- pre-tagging of drugs and genes to submit alongside a prompt. 
- support for multiple interactions per one abstract or block of text
- support for chunking sections of a PMID (Abstract, Results, Conclusion)
- support for Pydantic model level sanity checks to reduce LLM usage where not needed (may involve pre-tagging or deterministic rule sets that would provide a `run_llm? TRUE FALSE` type evaluation)

In [8]:
MODEL_ID = "us.anthropic.claude-sonnet-4-6"
REGION_NAME = "us-east-1"
PROFILE_NAME = "dev-account"
MAX_TOKENS = 1000

def build_llm_task_runner(
    model_id: str,
    region_name: str,
    profile_name: str,
    max_tokens: int,
    temperature: float,
) -> StructuredTaskRunner:
    """Build LLM interaction extraction task runner

    :param model_id: Bedrock model identifier.
    :param region_name: AWS region for the Bedrock runtime client.
    :param profile_name: AWS profile name.
    :param max_tokens: Maximum number of tokens to request from the model.
    :param temperature: Sampling temperature.
    :return: Configured structured task runner instance.
    """
    registry = build_empty_registry()
    registry.register(InteractionClassificationPrompt(version="v2"))
    llm_client = BedrockClaudeJsonClient(
        model_id=model_id,
        region_name=region_name,
        profile_name=profile_name,
        max_tokens=max_tokens,
        temperature=temperature,
    )
    cache = InMemoryCache()
    return StructuredTaskRunner(
        client=llm_client, prompt_registry=registry, cache=cache
    )

In [9]:
def classify_interaction(
    task_runner: StructuredTaskRunner,
    prompt: InteractionClassificationPrompt,
    context: str,
    candidate_drug: str,
    candidate_gene: str,
) -> InteractionClassificationResult:
    """Classify whether one candidate drug-gene pair is supported by biomedical text."""

    payload = prompt.build_payload(
        context=context,
        candidate_drug=candidate_drug,
        candidate_gene=candidate_gene,
    )

    try:
        task_result = task_runner.execute(
            prompt_name=prompt.name,
            prompt_version=prompt.version,
            payload=payload,
            response_model=InteractionClassificationResult,
        )

        return task_result

    except Exception as e:
        return InteractionClassificationResult(
            drug=candidate_drug,
            gene=candidate_gene,
            interaction=False,
            evidence=None,
            interaction_type=None,
            directionality=None,
            error_message=str(e),
        )

### Pre-Tagging


Description

In [10]:
df = pl.read_excel("../2026-06-08-test1-BCL2-only.xlsx") # file with pmids + context blocks

In [11]:
tagger = CompositeEntityTagger(
    BioBertEntityTagger(
        TaggerConfig(
            batch_size=16,
            include_drugs=True,
            include_genes=True,
            include_diseases=False,
            device=-1,
        )
    ),
    PubTator3ChemicalTagger.from_pubtator_file("chemical2pubtator3"),
)

pretagger = EntityPreTaggingService(
    tagger=tagger,
    normalizer=ViccNormalizer(
        cache=SQLiteNormalizerCache(".dgilit_normalizer_cache.sqlite")
    ),
)

In [12]:
contexts = (
    df["context"]
    .fill_null("")
    .cast(pl.Utf8)
    .to_list()
)
pmids = df["pmid"].cast(str).to_list() if "pmid" in df.columns else [None] * len(contexts)
block_ids = [str(i) for i in range(len(contexts))]

tagged_blocks = pretagger.tag_blocks(
    contexts=contexts,
    pmids=pmids,
    block_ids=block_ids,
)

Device set to use cpu


Tagging text batches:   0%|          | 0/4 [00:00<?, ?it/s]

Device set to use cpu


Tagging text batches:   0%|          | 0/4 [00:00<?, ?it/s]

Normalizing unique entities:   0%|          | 0/493 [00:00<?, ?entity/s]

In [13]:
def entity_display_name(e):
    if e.concept and e.concept.concept_label:
        return e.concept.concept_label

    if e.text:
        return e.text

    return None


def build_candidate_context(block):
    drugs = sorted({
        e.concept.concept_label
        for e in block.entities
        if e.entity_type == "drug"
        and e.concept
        and e.concept.concept_label
        and e.concept.concept_id
    })

    genes = sorted({
        e.concept.concept_label
        for e in block.entities
        if e.entity_type == "gene"
        and e.concept
        and e.concept.concept_label
        and e.concept.concept_id
    })

    return {
        "pmid": block.pmid,
        "drugs": drugs,
        "genes": genes,
        "context": block.context,
    }

candidate = build_candidate_context(tagged_blocks[0])
candidate

{'pmid': '14977850',
 'drugs': ['TRAIL', 'doxorubicin hydrochloride', 'etoposide', 'oxaliplatin'],
 'genes': ['TNFSF10'],
 'context': 'PURPOSE: Overexpression of antiapoptotic Bcl-2 family members has recently been related to resistance to chemo/radiotherapy in several human malignancies, particularly lymphomas. Hence, innovative approaches bypassing this resistance mechanism are required in the therapeutic approach. This study evaluated whether chemoresistance associated with Bcl-2 and Bcl-x(L) overexpression would be overcome by activating the death receptor pathway by tumor necrosis factor-related apoptosis-inducing ligand (TRAIL) in the Jurkat cell model\r\nEXPERIMENTAL DESIGN: We made use of genetically modified Jurkat cells to evaluate the effect of Bcl-2 or Bcl-x(L) overexpression on the cytotoxic effect produced by the anticancer drugs doxorubicin, etoposide, and oxaliplatin and TRAIL. Caspase activation was detected by cleavage of caspase-8 and -3. The mitochondrial transmambr

### LLM Calls


Description

In [14]:
# OLD
def entity_display_name(e):
    if e.concept and e.concept.concept_label and e.concept.concept_id:
        return e.concept.concept_label
    return None


def get_candidates(block):
    drugs = sorted({
        name
        for e in block.entities
        if e.entity_type == "drug"
        for name in [entity_display_name(e)]
        if name
    })

    genes = sorted({
        name
        for e in block.entities
        if e.entity_type == "gene"
        for name in [entity_display_name(e)]
        if name
    })

    return drugs, genes

In [15]:
# EXPERIMENTAL
def normalized_entity_name(e):
    if e.concept and e.concept.concept_label and e.concept.concept_id:
        return e.concept.concept_label
    return None


def raw_entity_name(e):
    if e.text:
        return e.text
    return None


def get_candidates(block):
    """Normalized candidates only."""
    drugs = sorted({
        name
        for e in block.entities
        if e.entity_type == "drug"
        for name in [normalized_entity_name(e)]
        if name
    })

    genes = sorted({
        name
        for e in block.entities
        if e.entity_type == "gene"
        for name in [normalized_entity_name(e)]
        if name
    })

    return drugs, genes


def get_raw_candidates(block):
    """Raw mentions regardless of normalization."""
    drugs = sorted({
        name
        for e in block.entities
        if e.entity_type == "drug"
        for name in [raw_entity_name(e)]
        if name
    })

    genes = sorted({
        name
        for e in block.entities
        if e.entity_type == "gene"
        for name in [raw_entity_name(e)]
        if name
    })

    return drugs, genes

block = tagged_blocks[0]

normalized_drugs, normalized_genes = get_candidates(block)
raw_drugs, raw_genes = get_raw_candidates(block)

candidate_drugs = sorted(
    set(normalized_drugs) | set(raw_drugs)
)

candidate_genes = sorted(
    set(normalized_genes) | set(raw_genes)
)

candidate_drugs



# DEBUG
block = tagged_blocks[0]

normalized_drugs, _ = get_candidates(block)
raw_drugs, _ = get_raw_candidates(block)

print("Normalized:")
print(normalized_drugs)

print("\nRaw:")
print(raw_drugs)

print("\nRaw only:")
print(sorted(set(raw_drugs) - set(normalized_drugs)))

# DEBUG

search_term = "PENTOXIFYLLINE"

hits = []

for i, block in enumerate(tagged_blocks):
    normalized_drugs, _ = get_candidates(block)
    raw_drugs, _ = get_raw_candidates(block)

    all_drugs = set(normalized_drugs) | set(raw_drugs)

    if any(search_term.lower() in drug.lower() for drug in all_drugs):
        hits.append({
            "index": i,
            "pmid": getattr(block, "pmid", None),
            "normalized_drugs": normalized_drugs,
            "raw_drugs": raw_drugs,
        })

print(f"Found {len(hits)} blocks containing '{search_term}'")

for hit in hits:
    print("\n" + "=" * 80)
    print(f"PMID: {hit['pmid']}")
    print(f"Block index: {hit['index']}")
    print(f"Normalized: {hit['normalized_drugs']}")
    print(f"Raw: {hit['raw_drugs']}")

Normalized:
['TRAIL', 'doxorubicin hydrochloride', 'etoposide', 'oxaliplatin']

Raw:
['TRAIL', 'doxorubicin', 'etoposide', 'oxaliplatin']

Raw only:
['doxorubicin']
Found 1 blocks containing 'PENTOXIFYLLINE'

PMID: 11836847
Block index: 22
Normalized: ['pentoxifylline']
Raw: ['PTX', 'pentoxifylline']


In [20]:
def run_experiments(
    tagged_blocks,
    temperatures,
    num_runs,
    prompt_version: str,
    use_raw_drug_candidates: bool = False,
    seed_gene: str | None = "BCL2",
):
    stored_runs = []

    for temp in temperatures:
        for run_idx in range(num_runs):
            task_runner = build_llm_task_runner(
                MODEL_ID,
                REGION_NAME,
                PROFILE_NAME,
                MAX_TOKENS,
                temp,
            )

            prompt = InteractionClassificationPrompt(version=prompt_version)

            print(f"Running temp={temp}, run={run_idx + 1}")

            results = []

            for block in tqdm(
                tagged_blocks,
                total=len(tagged_blocks),
                desc=f"T={temp}, run={run_idx + 1}",
                leave=False,
            ):
                normalized_drugs, normalized_genes = get_candidates(block)
                raw_drugs, raw_genes = get_raw_candidates(block)

                if use_raw_drug_candidates:
                    candidate_drugs = sorted(set(normalized_drugs) | set(raw_drugs))
                else:
                    candidate_drugs = normalized_drugs

                candidate_genes = normalized_genes

                if seed_gene:
                    candidate_genes_for_classification = [seed_gene]
                else:
                    candidate_genes_for_classification = candidate_genes

                if not candidate_drugs or not candidate_genes_for_classification:
                    results.append(
                        {
                            "pmid": block.pmid,
                            "block_id": block.block_id,
                            "skipped": True,
                            "skip_reason": "No drug and gene candidates",
                            "candidate_drugs": candidate_drugs,
                            "candidate_genes": candidate_genes,
                            "normalized_drugs": normalized_drugs,
                            "normalized_genes": normalized_genes,
                            "raw_drugs": raw_drugs,
                            "raw_genes": raw_genes,
                            "classifications": [],
                        }
                    )
                    continue

                classifications = []

                for candidate_drug, candidate_gene in product(
                    candidate_drugs,
                    candidate_genes_for_classification,
                ):
                    result = classify_interaction(
                        task_runner=task_runner,
                        prompt=prompt,
                        context=block.context,
                        candidate_drug=candidate_drug,
                        candidate_gene=candidate_gene,
                    )

                    classifications.append(result.model_dump())

                results.append(
                    {
                        "pmid": block.pmid,
                        "block_id": block.block_id,
                        "skipped": False,
                        "candidate_drugs": candidate_drugs,
                        "candidate_genes": candidate_genes_for_classification,
                        "normalized_drugs": normalized_drugs,
                        "normalized_genes": normalized_genes,
                        "raw_drugs": raw_drugs,
                        "raw_genes": raw_genes,
                        "num_candidate_pairs": len(candidate_drugs)
                        * len(candidate_genes_for_classification),
                        "classifications": classifications,
                    }
                )

            stored_runs.append(
                {
                    "run_idx": run_idx,
                    "prompt_version": prompt_version,
                    "temperature": temp,
                    "use_raw_drug_candidates": use_raw_drug_candidates,
                    "seed_gene": seed_gene,
                    "results": results,
                }
            )

            print(f"Done temp={temp}, run={run_idx + 1}")

    return stored_runs

In [ ]:
# DEBUG
for block in tagged_blocks:
    if str(block.pmid) == "11836847":
        normalized_drugs, _ = get_candidates(block)
        raw_drugs, _ = get_raw_candidates(block)

        print(normalized_drugs)
        print(raw_drugs)
        break

['pentoxifylline']
['PTX', 'pentoxifylline']


In [21]:
stored_runs = run_experiments(
    tagged_blocks=tagged_blocks,
    temperatures=[0],
    num_runs=1,
    prompt_version="v2",
    use_raw_drug_candidates=False,
    seed_gene="BCL2",
)

Running temp=0, run=1


T=0, run=1:   0%|          | 0/62 [00:00<?, ?it/s]

Done temp=0, run=1


NOTE: After running this once, temperature as a parameter really doesn't need to be included. Kinda just set it to 0 and forget about it tbh.

In [22]:
stored_runs



[{'run_idx': 0,
  'prompt_version': 'v2',
  'temperature': 0,
  'use_raw_drug_candidates': False,
  'seed_gene': 'BCL2',
  'results': [{'pmid': '14977850',
    'block_id': '0',
    'skipped': False,
    'candidate_drugs': ['TRAIL',
     'doxorubicin hydrochloride',
     'etoposide',
     'oxaliplatin'],
    'candidate_genes': ['BCL2'],
    'normalized_drugs': ['TRAIL',
     'doxorubicin hydrochloride',
     'etoposide',
     'oxaliplatin'],
    'normalized_genes': ['TNFSF10'],
    'raw_drugs': ['TRAIL', 'doxorubicin', 'etoposide', 'oxaliplatin'],
    'raw_genes': ['Bcl - 2',
     'Bcl - 2 family members',
     'Bcl - 2 family proteins',
     'Bcl - x ( L',
     'Bcl - x ( L )',
     'Caspase',
     'TRAIL',
     'caspase',
     'caspase - 8',
     'caspase - 8 and - 3',
     'death receptor',
     'tumor necrosis factor - related apoptosis - inducing ligand',
     'zVAD - fmk'],
    'num_candidate_pairs': 4,
    'classifications': [{'drug': 'TRAIL',
      'gene': 'BCL2',
      'interac

### Save

Save the output from a wags-llm run. After running through this whole process but I think its good.

In [23]:
def save_experiment_results(
    stored_runs,
    csv_path="2026-06-16-interaction_classifications-bcl2.csv",
    parquet_path="2026-06-16-interaction_classifications-bcl2.parquet",
):
    rows = []

    for run in stored_runs:
        for result in run["results"]:
            candidate_drugs = result.get("candidate_drugs", [])
            candidate_genes = result.get("candidate_genes", [])
            classifications = result.get("classifications", [])

            normalized_drugs = result.get("normalized_drugs", [])
            normalized_genes = result.get("normalized_genes", [])
            raw_drugs = result.get("raw_drugs", [])
            raw_genes = result.get("raw_genes", [])

            base_row = {
                "run_idx": run.get("run_idx"),
                "prompt_version": run.get("prompt_version"),
                "temperature": run.get("temperature"),
                "used_raw_drug_candidates": run.get(
                    "use_raw_drug_candidates",
                    False,
                ),
                "seed_gene": run.get("seed_gene"),

                "pmid": result.get("pmid"),
                "block_id": result.get("block_id"),
                "skipped": result.get("skipped", False),
                "skip_reason": result.get("skip_reason"),

                "candidate_drugs": ";".join(candidate_drugs),
                "candidate_genes": ";".join(candidate_genes),
                "candidate_drug_count": len(candidate_drugs),
                "candidate_gene_count": len(candidate_genes),
                "num_candidate_pairs": result.get("num_candidate_pairs", 0),

                "normalized_drugs": ";".join(normalized_drugs),
                "normalized_genes": ";".join(normalized_genes),
                "normalized_drug_count": len(normalized_drugs),
                "normalized_gene_count": len(normalized_genes),

                "raw_drugs": ";".join(raw_drugs),
                "raw_genes": ";".join(raw_genes),
                "raw_drug_count": len(raw_drugs),
                "raw_gene_count": len(raw_genes),

                "error_message": result.get("error_message"),
            }

            if not classifications:
                rows.append(
                    {
                        **base_row,
                        "drug": None,
                        "gene": None,
                        "interaction": None,
                        "interaction_type": None,
                        "directionality": None,
                        "evidence": None,
                    }
                )
                continue

            for classification in classifications:
                rows.append(
                    {
                        **base_row,
                        "drug": classification.get("drug"),
                        "gene": classification.get("gene"),
                        "interaction": classification.get("interaction"),
                        "interaction_type": classification.get("interaction_type"),
                        "directionality": classification.get("directionality"),
                        "evidence": classification.get("evidence"),
                        "error_message": classification.get("error_message")
                        or result.get("error_message"),
                    }
                )

    df_results = pl.DataFrame(rows)

    df_results.write_csv(csv_path)
    df_results.write_parquet(parquet_path)

    print(df_results.shape)
    return df_results

df_results = save_experiment_results(stored_runs)

df_results.head()

(124, 29)


run_idx,prompt_version,temperature,used_raw_drug_candidates,seed_gene,pmid,block_id,skipped,skip_reason,candidate_drugs,candidate_genes,candidate_drug_count,candidate_gene_count,num_candidate_pairs,normalized_drugs,normalized_genes,normalized_drug_count,normalized_gene_count,raw_drugs,raw_genes,raw_drug_count,raw_gene_count,error_message,drug,gene,interaction,interaction_type,directionality,evidence
i64,str,i64,bool,str,str,str,bool,str,str,str,i64,i64,i64,str,str,i64,i64,str,str,i64,i64,str,str,str,bool,str,str,str
0,"""v2""",0,false,"""BCL2""","""14977850""","""0""",false,null,"""TRAIL;doxorubicin hydrochlorid…","""BCL2""",4,1,4,"""TRAIL;doxorubicin hydrochlorid…","""TNFSF10""",4,1,"""TRAIL;doxorubicin;etoposide;ox…","""Bcl - 2;Bcl - 2 family members…",4,13,null,"""TRAIL""","""BCL2""",true,"""resistance modulation""","""inhibiting""",null
0,"""v2""",0,false,"""BCL2""","""14977850""","""0""",false,null,"""TRAIL;doxorubicin hydrochlorid…","""BCL2""",4,1,4,"""TRAIL;doxorubicin hydrochlorid…","""TNFSF10""",4,1,"""TRAIL;doxorubicin;etoposide;ox…","""Bcl - 2;Bcl - 2 family members…",4,13,null,"""doxorubicin hydrochloride""","""BCL2""",true,"""resistance""","""inhibitory""",null
0,"""v2""",0,false,"""BCL2""","""14977850""","""0""",false,null,"""TRAIL;doxorubicin hydrochlorid…","""BCL2""",4,1,4,"""TRAIL;doxorubicin hydrochlorid…","""TNFSF10""",4,1,"""TRAIL;doxorubicin;etoposide;ox…","""Bcl - 2;Bcl - 2 family members…",4,13,null,"""etoposide""","""BCL2""",true,"""resistance""","""inhibitory""",null
0,"""v2""",0,false,"""BCL2""","""14977850""","""0""",false,null,"""TRAIL;doxorubicin hydrochlorid…","""BCL2""",4,1,4,"""TRAIL;doxorubicin hydrochlorid…","""TNFSF10""",4,1,"""TRAIL;doxorubicin;etoposide;ox…","""Bcl - 2;Bcl - 2 family members…",4,13,null,"""oxaliplatin""","""BCL2""",true,"""resistance""","""inhibitory""",null
0,"""v2""",0,false,"""BCL2""","""16684279""","""1""",false,null,"""MPA;calcipotriene""","""BCL2""",2,1,2,"""MPA;calcipotriene""","""TP53""",2,1,"""MPA;calcipotriol;methylprednis…","""bcl - 2;ki - 67;p53""",3,3,null,"""MPA""","""BCL2""",true,"""expression modulation""","""activating""",null


In [25]:
pentox = df_results.filter(
    pl.col("candidate_drugs")
    .str.to_lowercase()
    .str.contains("pentoxifylline")
)

pentox.select([
    "pmid",
    "candidate_drugs",
    "drug",
    "gene",
    "interaction",
    "interaction_type",
    "directionality",
    "evidence",
])

pmid,candidate_drugs,drug,gene,interaction,interaction_type,directionality,evidence
str,str,str,str,bool,str,str,str
"""11836847""","""pentoxifylline""","""pentoxifylline""","""BCL2""",true,"""expression modulation""","""activating""",null
